In [1]:
# ------------------------------------------------------------------------------
# Setup and Path Validation
# ------------------------------------------------------------------------------
from app.core.tools.catalog_tool import catalog_lookup, CATALOG_PATH
import os
import json
from pathlib import Path
import sys

# Ensure we can import catalog_tool
HERE = Path.cwd().resolve()
print("Running from:", HERE)


def is_repo_root(p: Path) -> bool:
    # Prefer strong layout markers to avoid accidentally selecting /scripts as root
    return (
        (p / "pyproject.toml").exists()
        or (p / ".git").exists()
        or ((p / "app").exists() and (p / "scripts").exists() and (p / "data").exists())
        or ((p / "knowledge").exists() and (p / "data").exists())
        or (p / "main.py").exists()
    )

# Adjust this to point to your repo root so 'data/catalog.json' is reachable
PROJECT_ROOT = next(p for p in [HERE, *HERE.parents] if is_repo_root(p))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Catalog Path: {CATALOG_PATH}")
print(f"Catalog exists? {CATALOG_PATH.exists()}")

Running from: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/tests
Project Root: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot
Catalog Path: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/data/catalog.json
Catalog exists? True


In [2]:
# ------------------------------------------------------------------------------
# Inspect Catalog Data
# ------------------------------------------------------------------------------
# Let's see what we are working with
if CATALOG_PATH.exists():
    data = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
    # Note: your ingest script uses "items", tool uses "products"
    items = data.get("items", [])
    print(f"Total items in catalog: {len(items)}")

    # Check first 2 items to see structure
    for item in items[:2]:
        print(
            f"- ID: {item['id']} | Title: {item['title']} | Family: {item['family']}")
else:
    print("❌ ERROR: catalog.json not found. Run ingest_raw_catalog.ipynb first.")


Total items in catalog: 247
- ID: 205.112 | Title: Comando Connect - Repuesto De Climatizador Neil | Family: CLIMATIZADOR
- ID: 3216.05.2 | Title: Goma de Sensor - Repuesto Climatizadores  Neil | Family: CLIMATIZADOR


In [3]:
# ------------------------------------------------------------------------------
#  Helper Function for Pretty Printing Results
# ------------------------------------------------------------------------------

def test_query(query, family=None, k=3):
    print(f"\nSearching for: '{query}' | Family Filter: {family}")
    try:
        results = catalog_lookup(query, product_family=family, k=k)
        print(f"Found {results['count']} matches (showing top {k}):")
        for i, match in enumerate(results['matches'], 1):
            price = match.get('price', 'N/A')
            print(
                f"  {i}. [{match.get('id')}] {match.get('title')} - ${price} {results['currency']}")
    except Exception as e:
        print(f"❌ Search failed: {e}")

In [4]:
# ------------------------------------------------------------------------------
# 5) Functional Smoke Tests
# ------------------------------------------------------------------------------


# Test 1: Exact ID lookup
# Replace 'ID_HERE' with a real ID from your catalog.json
test_query("C101")

# Test 2: Title lookup (Fuzzy/Token match)
test_query("HDK 2200")

# Test 3: Family filtering (Should only return TPMS)
test_query("sensor", family="TPMS")

# Test 4: Broad search (Should return multiple items up to k=5)
test_query("aire", k=5)


Searching for: 'C101' | Family Filter: None
[{'id': 'C035', 'title': 'TPMS con conexión a encendedor + Puerto USB - Sensores externos - C101', 'description': 'TPMS C101 EXTERNO para autos y camionetas. Sensor externo con monitor de encendedor y salida USB. Mide presión y temperatura en tiempo real. Fácil instalación.', 'availability': 'in stock', 'condition': 'new', 'price': 78159.95, 'sale_price': 75034.0, 'link': 'https://neil.ar/p/tpms-con-conexion-a-encendedor-puerto-usb-sensores-externos-c101/cf6d6894-b1ae-421f-990d-5bca438c404d', 'image_link': 'https://d2eebw31vcx88p.cloudfront.net/neil/uploads-r/p/2d14206e006e3e93425887786990c53d8e7f36ce.jpg', 'brand': 'Neil', 'google_product_category': 'Accesorios Para Vehículos > Medidor De Presión Y Temperatura De Neumáticos (Tpms)', 'custom_label_1': '', 'family': 'TPMS'}, {'id': 'C036', 'title': 'TPMS con conexión a encendedor + Puerto USB - Sensores internos - C101', 'description': 'TPMS C-101: sensores internos, conexión directa al encen

In [6]:
# ------------------------------------------------------------------------------
# 6) Edge Case Testing
# ------------------------------------------------------------------------------

print("\n--- Edge Case Testing ---")

# Test: Case Insensitivity
test_query("tpms")
test_query("TPMS")

# Test: Non-existent product
test_query("Flux Capacitor 9000")

# Test: Empty Query (behavior check)
test_query("")


--- Edge Case Testing ---

Searching for: 'tpms' | Family Filter: None
[{'id': 'C019', 'title': 'TPMS para Utilitarios y Motorhome - Sensores Internos - C300', 'description': 'TPMS C300: sensores externos, ideal para camiones pequeños y motorhomes. Medición solar en tiempo real de presión y temperatura para 6 neumáticos.', 'availability': 'in stock', 'condition': 'new', 'price': 188301.0, 'sale_price': 188301.0, 'link': 'https://neil.ar/p/tpms-para-utilitarios-y-motorhome-sensores-internos-c300/59d7a94d-80c6-4be7-b930-bfa3ca4657ca', 'image_link': 'https://d2eebw31vcx88p.cloudfront.net/neil/uploads-r/p/b1e1e624653ef1dff5c582b17a1681d7b09fcbb8.jpg', 'brand': 'Neil', 'google_product_category': 'Accesorios Para Vehículos > Medidor De Presión Y Temperatura De Neumáticos (Tpms)', 'custom_label_1': '', 'family': 'TPMS'}, {'id': 'C022', 'title': 'Tpms para autos - Sensores Externos - C260', 'description': 'TPMS C260+ Externo: sensores fáciles de instalar, carga solar, alertas de presión y tem